[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mento-calc/mento/blob/main/docs/source/examples/shear_wall_summary_ACI_318-19.ipynb)

In [1]:
# Google Colab does not ship mento, so install it when running there.
# Guarded so that building the documentation never shells out to pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Shear Wall Summary with ACI 318-19

**Create input dataframe**

This example builds the two tables, sections and forces, directly in the cell; it is easier to read them from the sheets `Sections` and `Forces` of an Excel file with `from_excel`, as the User Guide explains.

Each (Level, Label) is a wall: one row in the sections table, and one row per load combination in the forces table.

In [2]:
from mento import MPa
from mento import ShearWallSummary, Concrete_ACI_318_19, SteelBar
import pandas as pd

conc = Concrete_ACI_318_19(name="H25", f_c=25 * MPa)
steel = SteelBar(name="ADN 420", f_y=420 * MPa)

# One row per wall, keyed by (Level, Label): its geometry and its mesh on each face.
sections = pd.DataFrame(
    {
        "Level": ["", "Level 1", "Level 2", "Level 1", "Level 2"],
        "Label": ["", "M1", "M1", "M2", "M2"],
        "t": ["cm", 20, 20, 20, 20],
        "lw": ["m", 3, 3, 2, 2],
        "hw": ["m", 3, 3, 3, 3],
        "cc": ["mm", 25, 25, 25, 25],
        "dbh": ["mm", 8, 8, 8, 8],
        "sh": ["cm", 20, 20, 20, 20],
        "dbv": ["mm", 12, 12, 12, 12],
        "sv": ["cm", 15, 15, 15, 15],
    }
)
# One row per load combination of a wall: Nx > 0 compression, Vz the in-plane shear; My is not used.
forces = pd.DataFrame(
    {
        "Level": [
            "",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 2",
            "Level 2",
            "Level 2",
            "Level 2",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 2",
            "Level 2",
            "Level 2",
            "Level 2",
        ],
        "Label": ["", "M1", "M1", "M1", "M1", "M1", "M1", "M1", "M1", "M2", "M2", "M2", "M2", "M2", "M2", "M2", "M2"],
        "Comb.": [
            "",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
        ],
        "Nx": ["kN", 0, 0, 0, -301, -150, 55.5, 282, -4.5, -240, -163, -17, 332, -150, 55.5, -163, 55.5],
        "Vz": ["kN", 264, 138, 123, 152, 32.3, 163, 19, 88.15, 61.2, 29, 47, 21, 32.3, 163, 29, 163],
        "My": ["kNm", -172, -90, -81, -234, 143, -278, 159, -97, -38, 60, -39, 46.13, 143, -278, 60, -278],
    }
)

wall_summary = ShearWallSummary(conc, steel, sections, forces)
# The sections table, as wall_summary.to_excel() writes it
wall_summary.sections_table

,Level,Label,t,lw,hw,cc,dbh,sh,dbv,sv
0,,,cm,m,m,mm,mm,cm,mm,cm
1,Level 1,M1,20,3,3,25,8,20,12,15
2,Level 2,M1,20,3,3,25,8,20,12,15
3,Level 1,M2,20,2,3,25,8,20,12,15
4,Level 2,M2,20,2,3,25,8,20,12,15


**Check all walls for shear**

In [3]:
wall_summary.check()

,Level,Label,t,lw,hw,Horiz. (each face),Vert. (each face),ρt,ρl,Comb.,Vu,Nu,ØVn,DCR,Warnings,Status
0,,,cm,m,m,,,,,,kN,kN,kN,,,
1,Level 1,M1,20.0,3.0,3.0,Ø8/20,Ø12/15,0.00251,0.00754,ELU 1,264.0,0.0,1037.5,0.254,-,✅
2,Level 2,M1,20.0,3.0,3.0,Ø8/20,Ø12/15,0.00251,0.00754,ELU 2,163.0,55.5,1037.5,0.157,-,✅
3,Level 1,M2,20.0,2.0,3.0,Ø8/20,Ø12/15,0.00251,0.00754,ELU 1,61.2,-240.0,527.3,0.116,-,✅
4,Level 2,M2,20.0,2.0,3.0,Ø8/20,Ø12/15,0.00251,0.00754,"ELU 2, ELU 4",163.0,55.5,691.7,0.236,-,✅


**See detailed shear results for all walls (all load combinations)**

In [4]:
wall_summary.shear_results()

,Label,Comb.,"ρt,min","ρt,req",ρt,"ρl,min",ρl,Vu,ØVc,ØVs,ØVn,"ØVn,max","Vu≤ØVn,max",Vu≤ØVn,DCR
0,,,,,,,,kN,kN,kN,kN,kN,,,
1,M1,ELU 1,0.0025,0.0025,0.00251,0.0025,0.00754,264.0,562.5,475.01,1037.51,1485.0,True,True,0.254
2,M1,ELU 2,0.0025,0.0025,0.00251,0.0025,0.00754,138.0,562.5,475.01,1037.51,1485.0,True,True,0.133
3,M1,ELU 3,0.0025,0.0025,0.00251,0.0025,0.00754,123.0,562.5,475.01,1037.51,1485.0,True,True,0.119
4,M1,ELU 4,0.0025,0.0025,0.00251,0.0025,0.00754,152.0,326.88,475.01,801.89,1485.0,True,True,0.19
5,M1,ELU 1,0.0025,0.0025,0.00251,0.0025,0.00754,32.3,354.78,475.01,829.79,1485.0,True,True,0.039
6,M1,ELU 2,0.0025,0.0025,0.00251,0.0025,0.00754,163.0,562.5,475.01,1037.51,1485.0,True,True,0.157
7,M1,ELU 3,0.0025,0.0025,0.00251,0.0025,0.00754,19.0,562.5,475.01,1037.51,1485.0,True,True,0.018
8,M1,ELU 4,0.0025,0.0025,0.00251,0.0025,0.00754,88.15,381.67,475.01,856.68,1485.0,True,True,0.103
9,M2,ELU 1,0.0025,0.0025,0.00251,0.0025,0.00754,61.2,210.65,316.67,527.32,990.0,True,True,0.116


**See detailed shear results for a single wall**

In [5]:
wall_summary.shear_results(index=1)

,Label,Comb.,"ρt,min","ρt,req",ρt,"ρl,min",ρl,Vu,ØVc,ØVs,ØVn,"ØVn,max","Vu≤ØVn,max",Vu≤ØVn,DCR
0,,,,,,,,kN,kN,kN,kN,kN,,,
1,M1,ELU 1,0.0025,0.0025,0.00251,0.0025,0.00754,264.0,562.5,475.01,1037.51,1485.0,True,True,0.254
2,M1,ELU 2,0.0025,0.0025,0.00251,0.0025,0.00754,138.0,562.5,475.01,1037.51,1485.0,True,True,0.133
3,M1,ELU 3,0.0025,0.0025,0.00251,0.0025,0.00754,123.0,562.5,475.01,1037.51,1485.0,True,True,0.119
4,M1,ELU 4,0.0025,0.0025,0.00251,0.0025,0.00754,152.0,326.88,475.01,801.89,1485.0,True,True,0.19


**Detailed results for a specific wall node**

In [6]:
wall_summary.nodes[0].shear_results_detailed()

===== SHEAR WALL DETAILED RESULTS =====
Materials                            Variable     Value  Unit
----------------------------------  ----------  -------  ------
Section Label                                        M1
Concrete strength                       fc           25  MPa
Steel reinforcement yield strength      fy          420  MPa
Normalweight concrete                   λ             1
Safety factor for shear                 Øv         0.75 

Geometry           Variable     Value  Unit
----------------  ----------  -------  ------
Wall thickness        t            20  cm
Wall length           lw          300  cm
Wall height           hw          300  cm
Aspect ratio        hw/lw           1
Gross shear area     Acv         6000  cm² 

Design forces     Variable     Value  Unit
---------------  ----------  -------  ------
Shear                Vu          264  kN 

Check                            Unit     Value  Min.       Max.  Ok?
------------------------------  ------  --

**Export results to Excel**

In [7]:
wall_summary.check().to_excel("Shear_wall_summary_results.xlsx", index=False)

**Export detailed results to Word**

In [8]:
wall_summary.results_detailed_doc(index=1)

✅ Results exported to Shear_Wall_Summary_ACI 318-19.docx
